# 08 · Análisis de errores (Fase 17)

**Pregunta.** ¿Dónde se equivoca el ensamble y qué característica falta? El objetivo no es
"mejorar el número" sino **identificar fenómenos** que orienten hipótesis nuevas.

Se analizan las predicciones **out-of-fold** del ensamble (el meta-modelo nunca ve el fold que
predice), sobre las mismas particiones agrupadas por clúster de near-duplicados.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import config as cfg
from src.config import set_seed

set_seed()
pd.set_option("display.width", 175)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 70)

def load_json(name):
    return json.loads((cfg.EXPERIMENTS / name).read_text(encoding="utf-8"))

clasicos = load_json("modelos_clasicos_resultados.json")
densos = load_json("e019_modelos_densos.json")
ensamble_mlp = load_json("e020_mlp_en_ensamble.json")
errores = load_json("fase17_error_analysis.json")
barrido_c = load_json("barrido_c_resultados.json")
print("artefactos cargados")

artefactos cargados


## 1. Matriz de confusión y desempeño por clase

In [2]:
matriz = pd.read_csv(cfg.TABLES / "fase17_matriz_confusion_normalizada.csv", index_col=0)
print("macro-F1 OOF del ensamble:", errores["oof_macro_f1"], "| accuracy:", errores["oof_accuracy"])
print("figura:", cfg.FIGURES / "fase17_matriz_confusion.png")
por_clase = pd.Series(errores["f1_por_clase"]).sort_values()
por_clase.to_frame("f1").round(4)

macro-F1 OOF del ensamble: 0.5656 | accuracy: 0.5658
figura: /home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/fase17_matriz_confusion.png


,f1
25-34,0.4492
18-24,0.4761
35-49,0.5189
weighted avg,0.5647
macro avg,0.5656
accuracy,0.5658
50-64,0.6199
13-17,0.6381
65-xx,0.6911


**OBSERVACIÓN.** Las clases extremas se clasifican bien (`13-17` 0,638; `65-xx` 0,691) y las
**intermedias concentran el error** (`25-34` 0,449; `18-24` 0,476).

**INTERPRETACIÓN.** Coincide con la hipótesis H2 del gate A: el registro informal separa bien los
extremos, pero los rangos contiguos comparten registro y léxico.

## 2. Pares más confundidos

In [3]:
confundidos = pd.read_csv(cfg.TABLES / "fase17_pares_confundidos.csv")
confundidos.head(8)

,real,predicho,casos,tasa_sobre_clase
0,13-17,18-24,6355,0.2202
1,25-34,18-24,6119,0.2159
2,18-24,13-17,5171,0.1803
3,18-24,25-34,4381,0.1528
4,35-49,25-34,4054,0.1436
5,35-49,50-64,4011,0.1421
6,25-34,35-49,3530,0.1245
7,65-xx,50-64,3455,0.1272


**OBSERVACIÓN.** La confusión es **casi toda entre clases adyacentes**: 18-24→25-34 (15,3%),
35-49→25-34 (14,4%), 35-49→50-64 (14,2%), 25-34→35-49 (12,5%), 65-xx→50-64 (12,7%).

**INTERPRETACIÓN.** El clasificador nominal desaprovecha la **estructura ordinal** del problema:
equivocarse por un rango vecino es mucho más frecuente que por dos o más.

**CONSECUENCIA.** Motiva el modelo ordinal auxiliar (E-015) y el uso del punto medio del rango.

## 3. Error por tipo de tweet

In [4]:
cortes = pd.read_csv(cfg.TABLES / "fase17_error_por_cortes.csv")
print(cortes[cortes["corte"] == "longitud"].to_string(index=False))
print()
print(cortes[cortes["corte"].isin(["retuit", "near-duplicado"])].to_string(index=False))
print()
print("figura:", cfg.FIGURES / "fase17_error_longitud.png")

         bucket     n  error_rate    corte
Q1 (más cortos) 34651      0.5251 longitud
             Q2 33708      0.4664 longitud
             Q3 33448      0.4243 longitud
             Q4 39721      0.3800 longitud
Q5 (más largos) 27808      0.3716 longitud

bucket      n  error_rate          corte
     0 107741      0.4622         retuit
     1  61595      0.3854         retuit
 False 161940      0.4415 near-duplicado
  True   7396      0.2749 near-duplicado

figura: /home/carlos/Git/PLN/KAGGLE_AGE_2026_PUJ/reports/figures/fase17_error_longitud.png


**OBSERVACIÓN.** La longitud domina el error: 52,5% en el quintil más corto frente a 37,2% en el
más largo. Los retuits y los near-duplicados se clasifican mejor (38,5% y 27,5% de error).

**INTERPRETACIÓN.** Los tweets cortos no contienen suficiente señal textual; los duplicados, en
cambio, se memorizan (lo que también explica la brecha positiva CV↔LB).

**CONSECUENCIA.** Se enriquece el meta-modelo con las features densas (longitud incluida) para que
**condicione** la combinación según el tipo de tweet (E-018: +0,0036).

## 4. Ejemplos de error

In [5]:
ejemplos = pd.read_csv(cfg.TABLES / "fase17_ejemplos_error.csv")
ejemplos.head(9)[["real", "predicho", "n_chars", "texto"]]

,real,predicho,n_chars,texto
0,13-17,18-24,84,Uno de los lugares mas hermosos que he visitado. Mi iglesia. https...
1,13-17,18-24,69,Como tiene que ser ! como tiene q' ser.... Que? http://t.co/q81peD68
2,13-17,18-24,30,:D ?xd http://t.co/uRbVMiFuPf
3,25-34,18-24,82,"RT @edsheeran: 'I wasn't made for rain, where's the sun!?' https:/..."
4,25-34,18-24,76,RT @pixidixidoo: Si el messenger te saca se te cierra la ventanita...
5,25-34,18-24,96,Quien me quiere acompanar manana en un recorrido por la candelaria...
6,18-24,13-17,139,RT @mayraestuvoaqui: Que feo ser tan sensible porque casi siempre ...
7,18-24,13-17,51,RT @YellowMellowMG: VEIS?!? https://t.co/VPdjvuf4jy
8,18-24,13-17,33,I hate how much I still love you.


### Catálogo de fenómenos de error

| # | Fenómeno | Evidencia | Hipótesis derivada | Resultado |
|---|---|---|---|---|
| F1 | Confusión entre rangos contiguos | 5 de los 6 pares más confundidos son adyacentes | señal ordinal (E-015) | útil en el ensamble (+0,0003) |
| F2 | Tweets cortos sin señal | 52,5% de error en el quintil corto | meta-modelo condicionado por longitud (E-018) | **+0,0036** |
| F3 | Clases medias (18-24, 25-34) | F1 0,449-0,476 | más vocabulario temático/fandom (canal de hashtags, E-014) | +0,0019 |
| F4 | Identidad de cuentas mencionadas | quitar menciones cuesta −0,125 | canal dedicado de menciones (E-008) | **+0,0144** |
| F5 | Errores distintos entre familias de modelo | MLP 0,4071 aporta +0,0066 | diversidad para el ensamble (E-020) | **+0,0066** |

**Cierre del ciclo.** Cada fenómeno de error produjo al menos una hipótesis medible y cuatro de
ellas se convirtieron en mejoras verificadas del modelo final.